# 🦟 West Nile Virus Surveillance — Colorado {#west-nile-surveillance}

**AEDES | Advanced Early Disease Prediction and Exploration Service**

This notebook tracks West Nile Virus (WNV) neuroinvasive disease in Colorado using:
- CDC NNDSS annual case reports
- NASA POWER daily temperature and precipitation data
- iNaturalist citizen-science mosquito observations

**Primary vector**: *Culex tarsalis* (western encephalitis mosquito)  
**Peak transmission**: July–September  
**Key risk factor**: Drought + heat waves concentrate birds and mosquitoes at shared water sources

## Quick Links to Report Sections

- [Summary of the Analysis](#summary)
- [Annual Case Trends (2010–2024)](#annual-trends)
- [Recent Climate Conditions (90-Day Window)](#climate-conditions)
- [iNaturalist Vector Observations](#vector-observations)
- [Early Warning Summary](#early-warning)

### Summary of the Analysis {#summary}

This notebook analyzes West Nile Virus surveillance data to identify trends and patterns. The results include:

- **Key Findings**: Highlights of the data analysis.
- **Visualizations**: Graphs and charts summarizing the data.
- **Insights**: Actionable insights derived from the analysis.

The focus is on presenting the results clearly, with the option to expand and view the query details if needed.

In [ ]:
# Core setup for reproducible execution in local runs and CI exports
from pathlib import Path
import json
import datetime as dt

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

TODAY = dt.date.today().isoformat()

# Resolve project root whether notebook runs from repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "surveillance"


def load_json(filename: str):
    """Load a surveillance JSON file from common execution locations."""
    candidates = [
        DATA_DIR / filename,
        Path("data/surveillance") / filename,
        Path("../data/surveillance") / filename,
    ]
    for path in candidates:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    return None

raw_preview = load_json("wnv_colorado.json")
if raw_preview and raw_preview.get("data"):
    preview_df = pd.DataFrame(raw_preview["data"])
    print(f"Loaded WNV dataset with {len(preview_df)} annual records")
    display(preview_df.tail(5))
else:
    print("WNV dataset unavailable. Downstream cells will use built-in fallback data.")

,fetched,source,data
0,2026-05-18,CDC NNDSS (historical),"{'year': 2010, 'state': 'Colorado', 'neuroinva..."
1,2026-05-18,CDC NNDSS (historical),"{'year': 2011, 'state': 'Colorado', 'neuroinva..."
2,2026-05-18,CDC NNDSS (historical),"{'year': 2012, 'state': 'Colorado', 'neuroinva..."
3,2026-05-18,CDC NNDSS (historical),"{'year': 2013, 'state': 'Colorado', 'neuroinva..."
4,2026-05-18,CDC NNDSS (historical),"{'year': 2014, 'state': 'Colorado', 'neuroinva..."


## 1. Annual Case Trends (2010–2024) {#annual-trends}

In [6]:
raw = load_json('wnv_colorado.json')

if raw and raw.get('data'):
    df_wnv = pd.DataFrame(raw['data'])
    source_label = raw.get('source', 'CDC NNDSS')
    print(f'Source: {source_label}')
else:
    # Built-in sample data (CDC published historical values)
    print('Using built-in historical data (no data file found)')
    source_label = 'CDC NNDSS (built-in)'
    df_wnv = pd.DataFrame({
        'year':           [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
        'neuroinvasive':  [  51,   20,  130,   14,   43,   72,   14,    5,   15,   10,   10,    8,   16,    6,   12],
        'deaths':         [   3,    1,    9,    0,    2,    3,    1,    0,    0,    0,    1,    0,    0,    0,    0],
    })

print(f'Records: {len(df_wnv)}')
print(f'Total neuroinvasive cases (all years): {df_wnv["neuroinvasive"].sum()}')
print(f'Total deaths (all years): {df_wnv["deaths"].sum()}')
print(f'Peak year: {df_wnv.loc[df_wnv["neuroinvasive"].idxmax(), "year"]} ({df_wnv["neuroinvasive"].max()} cases)')
df_wnv.tail(5)

NameError: name 'load_json' is not defined

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Colorado West Nile Virus Surveillance', fontsize=15, fontweight='bold', y=0.98)

# Neuroinvasive cases
colors = ['#e53e3e' if y == df_wnv.loc[df_wnv['neuroinvasive'].idxmax(), 'year'] else '#3182ce'
          for y in df_wnv['year']]
axes[0].bar(df_wnv['year'], df_wnv['neuroinvasive'], color=colors, alpha=0.85, zorder=3)
axes[0].set_ylabel('Neuroinvasive Cases', fontsize=11)
axes[0].set_title('Neuroinvasive Disease Cases by Year', fontsize=12)
axes[0].grid(axis='y', alpha=0.3, zorder=0)
axes[0].annotate('2012 outbreak\n(130 cases)', xy=(2012, 130), xytext=(2013.5, 120),
                 arrowprops=dict(arrowstyle='->', color='#e53e3e'),
                 fontsize=9, color='#e53e3e')

# Deaths
axes[1].bar(df_wnv['year'], df_wnv['deaths'], color='#742a2a', alpha=0.75, zorder=3)
axes[1].set_ylabel('Deaths', fontsize=11)
axes[1].set_title('WNV Deaths by Year', fontsize=12)
axes[1].grid(axis='y', alpha=0.3, zorder=0)
axes[1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].set_xlabel('Year', fontsize=11)

plt.tight_layout()
plt.savefig('wnv_annual_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Source: {source_label}')

## 2. Recent Climate Conditions (90-Day Window) {#climate-conditions}

In [7]:
raw_climate = load_json('climate_colorado_90d.json')

if raw_climate and raw_climate.get('data') and len(raw_climate['data']) > 0:
    df_clim = pd.DataFrame(raw_climate['data'])
    df_clim['date'] = pd.to_datetime(df_clim['date'], format='%Y%m%d', errors='coerce')
    df_clim = df_clim.dropna(subset=['date', 'temp_c'])

    # NASA POWER uses -999 as missing sentinel; drop impossible values.
    df_clim = df_clim[df_clim['temp_c'] > -80].copy()
    if 'precip_mm' in df_clim.columns:
        df_clim.loc[df_clim['precip_mm'] <= -900, 'precip_mm'] = pd.NA

    climate_source = raw_climate.get('source', 'NASA POWER')
    print(f'Climate data: {len(df_clim)} valid days from {climate_source}')
    print(f'Date range: {df_clim["date"].min().date()} to {df_clim["date"].max().date()}')
    print(f'Mean temp (°C): {df_clim["temp_c"].mean():.1f}')
    days_above_18 = int((df_clim['temp_c'] > 18).sum())
    print(f'Days above 18°C (WNV transmission threshold): {days_above_18}')
    have_climate = True
else:
    print('Climate API unavailable — showing transmission threshold reference only')
    have_climate = False

NameError: name 'load_json' is not defined

In [ ]:
if have_climate:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    fig.suptitle('Colorado Climate Conditions — Last 90 Days (Denver)', fontsize=13, fontweight='bold')

    # Temperature with WNV threshold line
    axes[0].plot(df_clim['date'], df_clim['temp_c'], color='#dd6b20', linewidth=1.5, label='Daily temp (°C)')
    axes[0].axhline(18, color='#e53e3e', linestyle='--', linewidth=1.2, label='WNV transmission threshold (18°C)')
    axes[0].fill_between(df_clim['date'], df_clim['temp_c'], 18,
                         where=df_clim['temp_c'] > 18, alpha=0.15, color='#e53e3e', label='Above threshold')
    axes[0].set_ylabel('Temperature (°C)', fontsize=10)
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # Precipitation
    if 'precip_mm' in df_clim.columns:
        axes[1].bar(df_clim['date'], df_clim['precip_mm'].fillna(0),
                    color='#3182ce', alpha=0.7, width=0.8, label='Precip (mm)')
        axes[1].set_ylabel('Precipitation (mm)', fontsize=10)
        axes[1].legend(fontsize=9)
        axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xlabel('Date', fontsize=10)

    plt.tight_layout()
    plt.savefig('wnv_climate_90d.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Source: {climate_source}')
else:
    print('Skipping climate chart (data unavailable)')

## 3. iNaturalist Vector Observations {#vector-observations}

In [ ]:
raw_inat = load_json('inaturalist_mosquitoes_colorado.json')

if raw_inat and raw_inat.get('data') and len(raw_inat['data']) > 0:
    df_inat = pd.DataFrame(raw_inat['data'])
    df_inat['observed_on'] = pd.to_datetime(df_inat['observed_on'], errors='coerce')
    df_inat = df_inat.dropna(subset=['observed_on'])

    print(f'iNaturalist mosquito observations (Colorado): {len(df_inat)}')
    print(f'Source: {raw_inat.get("source", "iNaturalist")}')
    if 'taxon' in df_inat.columns:
        print('\nTop species observed:')
        print(df_inat['taxon'].value_counts().head(8).to_string())

    # Monthly distribution
    df_inat['month'] = df_inat['observed_on'].dt.month
    monthly = df_inat.groupby('month').size().reindex(range(1, 13), fill_value=0)

    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(month_names, monthly.values, color='#2f855a', alpha=0.8)
    ax.set_title('iNaturalist Mosquito Observations by Month (Colorado)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Observations')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('inat_mosquitoes_monthly.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nFetched: {raw_inat.get("fetched", "unknown")}')
else:
    print('No iNaturalist data available (API unavailable or no observations returned)')

## 4. Early Warning Summary {#early-warning}

In [ ]:
import datetime
import calendar

current_month = datetime.date.today().month
current_month_name = calendar.month_name[current_month]
year_now = datetime.date.today().year

# Ensure required datasets exist even if user runs this cell directly.
if 'df_wnv' not in globals() or df_wnv is None or df_wnv.empty:
    raw = load_json('wnv_colorado.json')
    if raw and raw.get('data'):
        df_wnv = pd.DataFrame(raw['data'])
    else:
        df_wnv = pd.DataFrame(columns=['year', 'neuroinvasive', 'deaths'])

have_climate = 'df_clim' in globals() and isinstance(df_clim, pd.DataFrame) and (not df_clim.empty) and ('temp_c' in df_clim.columns)

# Seasonal risk by month (Colorado climatology context)
monthly_risk = {
    1: ('Low', '#48bb78'), 2: ('Low', '#48bb78'), 3: ('Low', '#48bb78'),
    4: ('Low', '#48bb78'), 5: ('Low', '#48bb78'), 6: ('Moderate', '#ed8936'),
    7: ('High', '#e53e3e'), 8: ('High', '#e53e3e'), 9: ('Moderate', '#ed8936'),
    10: ('Low', '#48bb78'), 11: ('Low', '#48bb78'), 12: ('Low', '#48bb78'),
}
risk_level, _ = monthly_risk[current_month]

# Current-season signal (evergreen weekly file)
raw_season = load_json(f'{year_now}_season_ytd.json')
wnv_ytd_cases = None
lyme_ytd_cases = None
baseline_ytd_wnv = None
if raw_season and raw_season.get('data'):
    season_df = pd.DataFrame(raw_season['data'])
    if 'wnv_cases' in season_df.columns:
        wnv_ytd_cases = int(pd.to_numeric(season_df['wnv_cases'], errors='coerce').fillna(0).sum())
    if 'lyme_cases' in season_df.columns:
        lyme_ytd_cases = int(pd.to_numeric(season_df['lyme_cases'], errors='coerce').fillna(0).sum())
    baseline_ytd_wnv = raw_season.get('historical_baseline_2024', {}).get('ytd_through_may', {}).get('wnv')

print('=' * 60)
print(f'  AEDES Early Warning Summary — {current_month_name} {year_now}')
print('=' * 60)
print(f'  Seasonal WNV Risk Level : {risk_level}')
print('  Peak transmission months: July-September')
print('  Primary vector          : Culex tarsalis')
print('  Key amplifying hosts    : Corvids, house finches')
print()

if have_climate:
    recent = df_clim.sort_values('date')['temp_c'].tail(14)
    recent_mean_temp = recent.mean() if len(recent) else float('nan')
    if pd.notna(recent_mean_temp):
        print(f'  14-day mean temperature : {recent_mean_temp:.1f}°C')
        print('  ✓ Temperature above WNV threshold (18°C)' if recent_mean_temp > 18 else '  ✓ Temperature below WNV threshold (18°C)')
    else:
        print('  14-day mean temperature : Data unavailable')
else:
    print('  14-day mean temperature : Data unavailable')

print()
print('  What this tells you right now:')
if wnv_ytd_cases is not None:
    print(f'  - {year_now} YTD WNV cases: {wnv_ytd_cases} (provisional)')
    if baseline_ytd_wnv is not None:
        trend = 'higher than' if wnv_ytd_cases > baseline_ytd_wnv else ('lower than' if wnv_ytd_cases < baseline_ytd_wnv else 'equal to')
        print(f'  - Compared with 2024 YTD through May ({baseline_ytd_wnv}), activity is {trend} baseline.')
else:
    print(f'  - {year_now} YTD WNV cases are not yet available from the weekly feed.')

if lyme_ytd_cases is not None:
    print(f'  - {year_now} YTD Lyme cases: {lyme_ytd_cases} (tick activity context)')

if not df_wnv.empty and {'neuroinvasive', 'deaths', 'year'}.issubset(df_wnv.columns):
    print(f'  - Historical context (2010-2024): {int(df_wnv["neuroinvasive"].sum())} neuroinvasive cases, {int(df_wnv["deaths"].sum())} deaths.')
    print(f'  - Most recent finalized annual data year: {int(df_wnv["year"].max())}.')

print('=' * 60)
print('  Data sources: CDC NNDSS | CDC provisional weekly | NASA POWER | iNaturalist')
print(f'  Generated   : {datetime.date.today()}')

  AEDES Early Warning Summary — May
  Seasonal WNV Risk Level : Low
  Peak transmission months: July–September
  Primary vector          : Culex tarsalis
  Key amplifying hosts    : Corvids, house finches

  14-day mean temperature : Data unavailable



NameError: name 'df_wnv' is not defined